# FedProx PyTorch MNIST Tutorial using Workflow API
This notebook sets up a distributed training federation which runs the `FedProx`[https://arxiv.org/abs/1812.06127] algorithm using OpenFL's  `Workflow API`[https://openfl.readthedocs.io/en/latest/about/features_index/workflowinterface.html] locally using a `LocalRuntime`[https://openfl.readthedocs.io/en/latest/about/features_index/workflowinterface.html#runtimes] - scalable to a federated setting in the future.


Import the relevant libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.utils
import torch.utils.data
import torchvision
import torchvision.transforms as transforms

from openfl.utilities.optimizers.torch.fedprox import FedProxAdam

from openfl.experimental.workflow.interface import FLSpec, Aggregator, Collaborator
from openfl.experimental.workflow.runtime import LocalRuntime
from openfl.experimental.workflow.placement import aggregator, collaborator

Define the model:

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3)
        self.fc1 = nn.Linear(32 * 5 * 5, 32)
        self.fc2 = nn.Linear(32, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0),-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

Set up the dataset:

In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

mnist_train = torchvision.datasets.MNIST(
    "./files/",
    train=True,
    download=True,
    transform=transform,
)

mnist_test = torchvision.datasets.MNIST(
    "./files/",
    train=False,
    download=True,
    transform=transform,
)

class CustomDataset(torch.utils.data.Dataset):
    """Dataset enumeration as tensors"""
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        return image, label

The next step is setting up the participants, an `Aggregator` and a few `Collaborator`s which will train the model, partition the dataset between the collaborators, and pass them to the appropriate runtime environment (in our case, a `LocalRuntime`).


In [ ]:
def one_hot(labels, classes):
    return np.eye(classes)[labels]

# Setup participants
aggregator_ = Aggregator()
aggregator_.private_attributes = {}

# Setup collaborators with private attributes
collaborator_names = [f'collaborator{i}' for i in range(4)]
collaborators = [Collaborator(name=name) for name in collaborator_names]
batch_size_train = 1024
batch_size_test = 1024
log_interval = 10

for idx, collaborator_ in enumerate(collaborators):
    train_images, train_labels = mnist_train.train_data, np.array(mnist_train.train_labels)
    train_images = torch.from_numpy(np.expand_dims(train_images, axis=1)).float()
    train_labels = one_hot(train_labels, 10)

    valid_images, valid_labels = mnist_test.test_data, np.array(mnist_test.test_labels)
    valid_images = torch.from_numpy(np.expand_dims(valid_images, axis=1)).float()

    collaborator_.private_attributes = {
            'train_loader': torch.utils.data.DataLoader(
                CustomDataset(train_images[idx::len(collaborators)], 
                              train_labels[idx::len(collaborators)]), 
                              batch_size=batch_size_train, 
                              shuffle=True),
            'test_loader': torch.utils.data.DataLoader(
                CustomDataset(valid_images[idx::len(collaborators)], 
                              valid_labels[idx::len(collaborators)]), 
                              batch_size=batch_size_test, 
                              shuffle=True)
    }

local_runtime = LocalRuntime(aggregator=aggregator_, collaborators=collaborators, backend='single_process')


Define an aggregation algorithm, optimizer and a loss function:

In [ ]:
# Aggregation algorithm
def FedAvg(models, weights=None):
    new_model = models[0]
    new_state_dict = dict()
    for key in new_model.state_dict().keys():
        new_state_dict[key] = torch.from_numpy(np.average([model.state_dict()[key].numpy() for model in models],
                                           axis=0, 
                                           weights=weights))

    new_model.load_state_dict(new_state_dict)
    return new_model

def get_optimizer(model):
    return FedProxAdam(model.parameters(), lr=1e-3, mu=0.01)

def cross_entropy(output, target):
    """Binary cross-entropy loss function"""
    return F.binary_cross_entropy_with_logits(input=output,target=target.float())

Set up work to be executed by the aggregator and the collaborators by extending `FLSpec`:

In [ ]:
class FederatedFlow(FLSpec):
    def __init__(self, model=None, optimizer=None, rounds=10, **kwargs):
        super().__init__(**kwargs)
        self.model = model
        self.optimizer = optimizer
        self.rounds = rounds
        self.loss = 0.

    @aggregator
    def start(self):
        print(f'Performing initialization for model')
        self.collaborators = self.runtime.collaborators
        self.current_round = 0
        self.next(self.aggregated_model_validation, foreach='collaborators')

    def compute_accuracy(self, data_loader):
        self.model.eval()
        test_loss = 0
        correct = 0
        with torch.no_grad():
            for data, target in data_loader:
                output = self.model(data)
                test_loss += F.cross_entropy(output, target, size_average=False).item()
                pred = output.data.max(1, keepdim=True)[1]
                correct += pred.eq(target.data.view_as(pred)).sum()

        test_loss /= len(data_loader.dataset)
        print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(data_loader.dataset),
        100. * correct / len(data_loader.dataset)))
        accuracy = float(correct / len(data_loader.dataset))
        return accuracy

    @collaborator
    def aggregated_model_validation(self):
        print(f'Performing aggregated model validation for collaborator {self.input}, model: {id(self.model)}')
        self.agg_validation_score = self.compute_accuracy(self.test_loader)
        self.next(self.train)

    @collaborator
    def train(self):
        # Log after processing a quarter of the samples
        log_threshold = .25

        self.model.train()
        self.optimizer = get_optimizer(self.model)
        
        # Set old weights ONCE at the beginning of training
        # This sets the reference weights to the global model weights 
        # received from the aggregator, implementing FedProx correctly
        self.optimizer.set_old_weights([p.clone().detach() for p in self.model.parameters()])
        
        for batch_idx, (data, target) in enumerate(self.train_loader):
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            
            self.optimizer.step()

            if (len(data) * batch_idx) / len(self.train_loader.dataset) >= log_threshold:
                print('Train Epoch: [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                    batch_idx * len(data), len(self.train_loader.dataset),
                    100. * batch_idx / len(self.train_loader), loss.item()))
                self.loss = loss.item()
                log_threshold += .25
                torch.save(self.model.state_dict(), 'model.pth')
                torch.save(self.optimizer.state_dict(), 'optimizer.pth')
            
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        print(f'Performing local model validation for collaborator {self.input}')
        self.local_validation_score = self.compute_accuracy(self.test_loader)
        print(
            f'Done with local model validation for collaborator {self.input}, Accuracy: {self.local_validation_score}')
        self.next(self.join)

    @aggregator
    def join(self, inputs):
        self.model = FedAvg([input.model for input in inputs])
        self.optimizer = inputs[0].optimizer
        self.current_round += 1

        self.average_loss = sum(input.loss for input in inputs) / len(inputs)
        self.aggregated_model_accuracy = sum(
            input.agg_validation_score for input in inputs) / len(inputs)
        self.local_model_accuracy = sum(
            input.local_validation_score for input in inputs) / len(inputs)
        print(f'Average aggregated model accuracy = {self.aggregated_model_accuracy}')
        print(f'Average training loss = {self.average_loss}')
        print(f'Average local model validation values = {self.local_model_accuracy}')

        if self.current_round < self.rounds:
            self.next(self.aggregated_model_validation, foreach='collaborators')
        else:
            self.next(self.end)

    @aggregator
    def end(self):
        print(f'Flow ended')

Finally, run the federation:

In [ ]:
model = Net()
flflow = FederatedFlow(model, get_optimizer(model), rounds=3, checkpoint=False)
flflow.runtime = local_runtime
flflow.run()

## Model Comparison with Different Mu Values
Now let's check if trained models with the same seeding but different mu values produce the same results. We'll train multiple models with different mu values and then compare their weights.

In [ ]:
import copy
import random
import numpy as np

# Function to set seed for reproducibility
def set_seed(seed):
    """Set seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Function to get optimizer with different mu values
def get_optimizer_with_mu(model, mu_value):
    """Get FedProxAdam optimizer with specific mu value"""
    return FedProxAdam(model.parameters(), lr=1e-3, mu=mu_value)

# Function to compare model weights
def compare_models(models, model_names):
    """Compare weights between different models"""
    print("Comparing model weights:")
    
    # Compare weights between each pair of models
    for i in range(len(models)):
        for j in range(i+1, len(models)):
            model1 = models[i]
            model2 = models[j]
            name1 = model_names[i]
            name2 = model_names[j]
            
            print(f"\nComparing {name1} and {name2}:")
            
            # Compare each layer's weights
            all_equal = True
            max_diff = 0.0
            
            for (p1, p2) in zip(model1.parameters(), model2.parameters()):
                # Check if parameters are equal
                if not torch.allclose(p1, p2, atol=1e-5):
                    all_equal = False
                    # Calculate maximum difference
                    diff = torch.max(torch.abs(p1 - p2)).item()
                    max_diff = max(max_diff, diff)
            
            if all_equal:
                print(f"Models {name1} and {name2} have identical weights")
            else:
                print(f"Models {name1} and {name2} have different weights")
                print(f"Maximum difference in weights: {max_diff:.6f}")

# Function to train a model with specific mu value
def train_model(model, mu_value, seed, epochs=1):
    """Train a model with specific mu value and seed"""
    set_seed(seed)  # Set seed for reproducibility
    
    # Create a copy of the model
    model_copy = copy.deepcopy(model)
    
    # Get optimizer with specific mu value
    optimizer = get_optimizer_with_mu(model_copy, mu_value)
    
    # Training loop
    model_copy.train()
    
    # Get a small dataset for quick testing
    train_images, train_labels = mnist_train.train_data[:1000], np.array(mnist_train.train_labels[:1000])
    train_images = torch.from_numpy(np.expand_dims(train_images, axis=1)).float()
    train_labels = one_hot(train_labels, 10)
    
    train_dataset = CustomDataset(train_images, train_labels)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    # CRITICAL FIX: Set the old weights ONCE at the beginning, before training
    # This properly implements FedProx by setting the reference weights
    # to the initial model weights (simulating the global model)
    optimizer.set_old_weights([p.clone().detach() for p in model_copy.parameters()])
    
    for epoch in range(epochs):
        running_loss = 0.0
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model_copy(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            
            # REMOVED: Don't call set_old_weights here - this was causing the issue
            optimizer.step()
            
            running_loss += loss.item()
            
        print(f"Epoch {epoch+1}, Mu={mu_value}, Loss: {running_loss/len(train_loader):.6f}")
    
    return model_copy

In [ ]:
# Define mu values to test
mu_values = [0.0, 0.01, 0.1, 0.5]
seed = 42  # Fixed seed for reproducibility
models = []
model_names = []

print("Training models with different mu values but same seed...")

# Train models with different mu values
for mu in mu_values:
    model_name = f"Model_mu{mu}"
    print(f"\nTraining {model_name} with mu={mu}...")
    
    # Initialize a new model
    base_model = Net()
    
    # Train the model with current mu value
    trained_model = train_model(base_model, mu, seed, epochs=2)
    
    # Save the model and name
    models.append(trained_model)
    model_names.append(model_name)

# Compare trained models
compare_models(models, model_names)

In [ ]:
# Visualize a few sample predictions to qualitatively check differences
def visualize_predictions(models, model_names):
    """Visualize predictions from different models on the same test samples"""
    # Get a few test samples
    test_images, test_labels = mnist_test.test_data[:5], mnist_test.test_labels[:5]
    test_images = torch.from_numpy(np.expand_dims(test_images, axis=1)).float()
    
    print("Predictions from different models:")
    for i, (image, label) in enumerate(zip(test_images, test_labels)):
        print(f"\nSample {i+1}, True label: {label}")
        
        # Make predictions with each model
        for model, name in zip(models, model_names):
            model.eval()
            with torch.no_grad():
                output = model(image.unsqueeze(0))
                pred = output.argmax(dim=1, keepdim=True).item()
                confidence = torch.nn.functional.softmax(output, dim=1).max().item()
                print(f"  {name}: Predicted {pred} with confidence {confidence:.4f}")

# Visualize predictions
visualize_predictions(models, model_names)

## Analysis of the Results

The results above show how different mu values in the FedProx algorithm affect model training. The mu parameter controls the strength of the proximal term, which penalizes the local model for deviating too much from the global model.

If the trained models with different mu values have identical weights, it would suggest that the mu parameter isn't having any effect on the training process with the current configuration. However, if they have different weights, it confirms that the mu parameter is working as expected - different values lead to different optimization paths.

The weight differences and prediction differences provide insights into how much the mu parameter affects the training process and final model behavior.

## Comparing Models in the Federated Setting with Different Mu Values

Let's now run a more comprehensive experiment to compare models trained in the federated setting with different mu values but the same random seed. This will help us understand how the mu parameter affects federated training.

In [ ]:
# Function to run federated training with specific mu value
def run_federated_with_mu(mu_value, rounds=3, seed=42):
    """Run federated training with specific mu value and seed"""
    print(f"\n\n--- Starting federated training with mu={mu_value}, seed={seed} ---\n")
    
    # Set global seed
    set_seed(seed)
    
    # Define optimizer getter function with specific mu
    def get_optimizer_with_mu_value(model):
        return FedProxAdam(model.parameters(), lr=1e-3, mu=mu_value)
    
    # Initialize model
    model = Net()
    
    # Create and run federated flow with the specified mu value
    flflow = FederatedFlow(model, get_optimizer_with_mu_value(model), rounds=rounds, checkpoint=False)
    flflow.runtime = local_runtime
    flflow.run()
    
    return model, flflow

# Let's save the original get_optimizer function to restore later
original_get_optimizer = get_optimizer

# Run experiment with different mu values
mu_values_federated = [0.0, 0.01, 0.1]
seed_value = 42
federated_models = []
federated_flows = []
federated_model_names = []

for mu in mu_values_federated:
    # Modify the get_optimizer function temporarily
    def get_optimizer_with_current_mu(model):
        return FedProxAdam(model.parameters(), lr=1e-3, mu=mu)
    
    # Replace the global function
    globals()['get_optimizer'] = get_optimizer_with_current_mu
    
    # Run federated training
    model_name = f"Federated_Model_mu{mu}"
    model, flow = run_federated_with_mu(mu, rounds=2, seed=seed_value)
    
    # Save results
    federated_models.append(model)
    federated_flows.append(flow)
    federated_model_names.append(model_name)

# Restore original get_optimizer function
globals()['get_optimizer'] = original_get_optimizer

# Compare the federated models
compare_models(federated_models, federated_model_names)

## Comparing Convergence and Performance Metrics

Now let's analyze how different mu values affect the convergence and final performance of the federated models.

In [ ]:
# Extract metrics from each flow for comparison
def extract_metrics_from_flow(flow):
    """Extract metrics from a completed flow for comparison"""
    rounds = flow.current_round
    metrics = {
        'aggregated_accuracy': [],
        'local_accuracy': [],
        'loss': []
    }
    
    # This is a simplification - in practice, we would extract metrics from the flow's history
    # Here we're just using the final metrics as a proxy
    metrics['aggregated_accuracy'].append(flow.aggregated_model_accuracy)
    metrics['local_accuracy'].append(flow.local_model_accuracy)
    metrics['average_loss'] = flow.average_loss
    
    return metrics

# Collect metrics from all flows
all_metrics = []
for i, flow in enumerate(federated_flows):
    mu = mu_values_federated[i]
    metrics = extract_metrics_from_flow(flow)
    metrics['mu'] = mu
    all_metrics.append(metrics)

# Print comparison of metrics
print("\nComparison of metrics across different mu values:")
print("-" * 60)
print(f"{'Mu Value':<10} | {'Final Aggregated Accuracy':<25} | {'Final Loss':<15}")
print("-" * 60)

for metrics in all_metrics:
    mu = metrics['mu']
    agg_acc = metrics['aggregated_accuracy'][-1] if metrics['aggregated_accuracy'] else 'N/A'
    loss = metrics['average_loss']
    print(f"{mu:<10.2f} | {agg_acc:<25.4f} | {loss:<15.6f}")

print("-" * 60)

## Testing Weight Divergence Directly 

The mu parameter in FedProx is designed to limit the divergence of local models from the global model. Let's directly measure this divergence for different mu values.

In [ ]:
def measure_weight_divergence(initial_model, trained_models, mu_values):
    """
    Measure how much each trained model has diverged from the initial model
    for different mu values
    """
    print("\nMeasuring weight divergence from initial model:")
    
    # Extract the initial model weights
    initial_weights = [p.clone().detach() for p in initial_model.parameters()]
    
    divergences = []
    for i, (model, mu) in enumerate(zip(trained_models, mu_values)):
        # Calculate divergence as the Euclidean distance between weight vectors
        total_divergence = 0
        for p_trained, p_initial in zip(model.parameters(), initial_weights):
            # Calculate squared Frobenius norm of the difference
            diff = p_trained - p_initial
            divergence = torch.norm(diff.flatten(), p=2).item()
            total_divergence += divergence
            
        divergences.append(total_divergence)
        print(f"Mu = {mu}: Total weight divergence = {total_divergence:.6f}")
    
    return divergences

# Create a fresh model to serve as the initial reference point
initial_model = Net()

# Measure divergence
divergences = measure_weight_divergence(initial_model, federated_models, mu_values_federated)

# We would expect models with higher mu to show less divergence from the initial point
# as the proximal term penalizes moving away from the global model

## Conclusion

These experiments help us understand how the `mu` parameter in FedProx affects model training and convergence. The `mu` parameter controls the strength of the proximal term, which penalizes the local model for deviating too much from the global model.

Key observations:

1. Different `mu` values should lead to different model weights if the proximal term is working as expected.
2. Higher `mu` values should result in less divergence between local models and the global model.
3. The effect of `mu` on accuracy and convergence speed can help determine the optimal value for a specific federated learning task.

This analysis provides insights into how to properly configure FedProx for different federated learning scenarios.

## Important Note on FedProx Implementation

This notebook includes a critical fix for how FedProx is applied in OpenFL:

### The FedProx Proximal Term

FedProx adds a proximal term to the objective function:

L(w) = F_i(w) + (μ/2) ||w - w^t||^2

Where:
- F_i(w) is the original loss function
- w^t is the global model from the previous round
- μ is the regularization parameter controlling how far local models can deviate

### Correct Implementation

The key insight is that `set_old_weights` should be called **once at the beginning** of each local training round, not before each optimization step:

```python
# At the beginning of local training:
self.optimizer = get_optimizer(self.model)
self.optimizer.set_old_weights([p.clone().detach() for p in self.model.parameters()])

# Then during training loop:
for batch_idx, (data, target) in enumerate(self.train_loader):
    self.optimizer.zero_grad()
    output = self.model(data)
    loss = F.cross_entropy(output, target)
    loss.backward()
    # DO NOT call set_old_weights here
    self.optimizer.step()
```

This ensures that the proximal term properly penalizes deviation from the global model, allowing different mu values to have the expected effect on model convergence.